# Sub-THz Blockage Prediction - LoRA Fine-tuning on Google Colab

Trains the two LoRA architectures (Arch A forecaster, Arch B classifier) on
**TimesFM 2.5** and reproduces the paper's metrics and figures on a Colab GPU.
The `src/` code is identical to `main`; this notebook orchestrates GPU checks,
Drive-backed output persistence, and resume-safe experiment loops.

### Before running
* `Runtime -> Change runtime type -> GPU`.
* Leave Internet access enabled so Colab can clone the repo, install packages,
  and download model weights.

Then run the cells top to bottom.


## 1. Check the GPU

If the current PyTorch build does not include kernels for the selected GPU's
compute capability, training would fail later with `no kernel image is available
for execution on the device`. This cell catches that before any long setup work.


In [ ]:
import torch

assert torch.cuda.is_available(), 'No GPU: enable a Colab GPU runtime first.'
p = torch.cuda.get_device_properties(0)
arch = f'sm_{p.major}{p.minor}'
supported = torch.cuda.get_arch_list()
print(f'{p.name} | {arch} | {p.total_memory/1e9:.1f} GB | torch {torch.__version__} (cuda {torch.version.cuda})')
print('torch supports:', supported)
if arch not in supported:
    raise SystemError(
        f'This torch build has no kernels for {p.name} ({arch}). '
        f'Switch to a Colab GPU whose architecture is listed above, restart, and re-run.'
    )
print(f'OK: {arch} kernels present.')


## 2. Clone the repo and install dependencies

The dataset is committed in the repo, so a shallow clone brings the code and
data. For a private repo, paste a GitHub token; leave `TOKEN` blank if it is
public.


In [ ]:
REPO = 'github.com/kaefcatcher/THz_blockage.git'
BRANCH = 'collab'
TOKEN = ''
PROJECT_DIR = '/content/THz_blockage'

import os

if not os.path.isdir('/content'):
    raise RuntimeError('This notebook is configured for Google Colab.')

url = f"https://{TOKEN + '@' if TOKEN else ''}{REPO}"
if not os.path.isdir(os.path.join(PROJECT_DIR, '.git')):
    !git clone --depth 1 --branch {BRANCH} {url} {PROJECT_DIR}
%cd {PROJECT_DIR}

!pip install -q -U 'transformers>=5.12' 'peft>=0.13' 'safetensors>=0.4' einops

import importlib, transformers, peft
importlib.reload(transformers)
try:
    from transformers import TimesFm2_5ModelForPrediction
    print(f'ready: transformers {transformers.__version__} | peft {peft.__version__} | TimesFM 2.5 OK')
except ImportError:
    print('transformers in this kernel is too old:', transformers.__version__)
    print('Restart the session, then re-run this cell.')
    raise


## 3. Output persistence

Mount Drive and link `outputs/` into it so checkpoints, CSVs, and figures survive
Colab disconnects. Set `USE_DRIVE = False` to keep outputs only in the ephemeral
runtime.


In [ ]:
import os, shutil

USE_DRIVE = True
DRIVE_OUT = '/content/drive/MyDrive/thz_blockage_outputs'

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(DRIVE_OUT, exist_ok=True)
    if os.path.isdir('outputs') and not os.path.islink('outputs'):
        for name in os.listdir('outputs'):
            src = os.path.join('outputs', name)
            dst = os.path.join(DRIVE_OUT, name)
            if os.path.isdir(src) and not os.path.islink(src):
                shutil.copytree(src, dst, dirs_exist_ok=True)
                shutil.rmtree(src)
            elif not os.path.exists(dst):
                shutil.move(src, dst)
        shutil.rmtree('outputs')
    elif os.path.lexists('outputs'):
        os.unlink('outputs')
    os.symlink(DRIVE_OUT, 'outputs')
else:
    os.makedirs('outputs', exist_ok=True)

os.makedirs('outputs/results', exist_ok=True)
print('outputs ->', os.path.realpath('outputs'))


## 4. Verify the data pipeline and LoRA setup

The first model load downloads the TimesFM 2.5 weights into the runtime cache.


In [ ]:
!python src/data.py
!python src/lora_common.py

## 5. GPU-aware memory flags

bfloat16 needs Ampere or newer GPUs (compute capability 8.0+). T4 and V100 use
fp32 with gradient checkpointing; Ampere GPUs use bf16.


In [ ]:
import torch

name = torch.cuda.get_device_name(0)
major = torch.cuda.get_device_capability()[0]
gb = torch.cuda.get_device_properties(0).total_memory / 1e9
bf16_ok = major >= 8
if bf16_ok and gb >= 24:
    CLI = '--bf16 --batch-size 64'
    TRAIN_KWARGS = dict(dtype='bf16', batch_size=64)
elif bf16_ok:
    CLI = '--bf16 --grad-checkpoint --batch-size 16 --accum-steps 4'
    TRAIN_KWARGS = dict(dtype='bf16', grad_checkpoint=True, batch_size=16, accum_steps=4)
else:
    CLI = '--grad-checkpoint --batch-size 32 --accum-steps 2'
    TRAIN_KWARGS = dict(grad_checkpoint=True, batch_size=32, accum_steps=2)
print(f'{name} cc{major}.x {gb:.0f}GB bf16={bf16_ok} -> {CLI}')


## 6. Train the canonical Arch A and Arch B checkpoints

Each command runs up to 30 epochs with early stopping. If a runtime runs out of
memory, lower `--batch-size` and raise `--accum-steps` to keep the effective
batch size similar.


In [ ]:
!python src/lora_arch_a.py {CLI}

In [ ]:
!python src/lora_arch_b.py --loss bce {CLI}
!python src/lora_arch_b.py --loss focal {CLI}

## 7. Main metrics table (Step 5)

In [ ]:
import sys; sys.path.insert(0, 'src')
import experiments
experiments.build_metrics_main(train_kwargs=TRAIN_KWARGS)

## 8. Data-efficiency experiment - resume-safe

The loop skips runs already present in the CSV and appends incrementally, so a
new Colab session can continue from the Drive-backed `outputs/` directory. For a
quick first pass, set `SEEDS = [0]` and/or trim `N_GRID`.


In [ ]:
import os, pandas as pd, sys
sys.path.insert(0, 'src')
import data, importlib, experiments; importlib.reload(experiments)
from experiments import run_one, N_GRID, SEEDS, TW

EFF = 'outputs/results/metrics_data_efficiency.csv'; os.makedirs('outputs/results', exist_ok=True)
done = set()
if os.path.exists(EFF):
    _d = pd.read_csv(EFF)
    done = {tuple(r) for r in _d[['arch','N_traces','seed']].itertuples(index=False, name=None)}

pool = data.get_train_pool_files()
for N in N_GRID:
    for seed in SEEDS:
        files = data.sample_traces_stratified(pool, N, seed=seed)
        for arch in ('A', 'B'):
            if (arch, N, seed) in done:
                print(f'skip arch={arch} N={N} seed={seed} (done)'); continue
            m = run_one(arch, files, seed=seed, loss='bce', train_kwargs=TRAIN_KWARGS)
            row = {'arch': arch, 'N_traces': N, 'seed': seed, 'Tw_ms': TW,
                   'acc': round(m['acc'],4), 'precision': round(m['precision'],4),
                   'recall': round(m['recall'],4), 'f1': round(m['f1'],4)}
            pd.DataFrame([row]).to_csv(EFF, mode='a', header=not os.path.exists(EFF), index=False)
            print(f"done arch={arch} N={N} seed={seed} f1={m['f1']:.4f}")
print('data-efficiency complete')

## 9. Diversity experiment - resume-safe (Step 7)

In [ ]:
import os, pandas as pd, sys, tempfile, pathlib
sys.path.insert(0, 'src')
import data, evaluate as ev, lora_arch_b
from experiments import _sample_config, _div_row, report_diversity, TW

DIV = 'outputs/results/metrics_diversity.csv'
pool = data.get_train_pool_files(); eval_files = data.get_eval_files()
theta = data.get_theta(); cfgs = sorted({data.meta_of(p)['set'] for p in eval_files})
done = set()
if os.path.exists(DIV):
    _d = pd.read_csv(DIV)
    done = {tuple(r) for r in _d[['condition','seed']].itertuples(index=False, name=None)}

for seed in [0, 1, 2]:
    conds = {'balanced': data.sample_traces_stratified(pool, 80, seed=seed),
             'homogeneous': _sample_config(pool, 1, 40, seed) + _sample_config(pool, 2, 40, seed)}
    for cond, files in conds.items():
        if (cond, seed) in done:
            print(f'skip {cond} seed={seed} (done)'); continue
        with tempfile.TemporaryDirectory() as tmp:
            model, _f1, _t = lora_arch_b.train(loss_kind='bce', train_files=files, seed=seed,
                                               save_dir=pathlib.Path(tmp), enforce_gate=False, **TRAIN_KWARGS)
            rows = [_div_row(cond, seed, 'overall', ev.evaluate(model, eval_files, TW, theta, 'arch_b'))]
            for c in cfgs:
                ef = data.files_for_configs(eval_files, [c])
                rows.append(_div_row(cond, seed, str(c), ev.evaluate(model, ef, TW, theta, 'arch_b')))
        pd.DataFrame(rows).to_csv(DIV, mode='a', header=not os.path.exists(DIV), index=False)
        print(f'done {cond} seed={seed}'); del model
report_diversity(pd.read_csv(DIV))

## 10. Figures (Step 8)

In [ ]:
!python src/figures.py --with-models
for f in ('data_efficiency_curve', 'pr_curve', 'metrics_table'):
    print('outputs/figures/' + f + '.pdf')

## 11. Save / export results

Outputs are a few MB: LoRA adapters, CSVs, and PDFs. Zip and download them after
the run, or keep using the Drive-backed `outputs/` directory for resuming.


In [ ]:
import shutil
from google.colab import files

dest = '/tmp/thz_outputs'
shutil.make_archive(dest, 'zip', 'outputs')
print('zip ->', dest + '.zip')
files.download(dest + '.zip')
